# Access SWOT Level 2 data with OPeNDAP

In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
# import pydap-specific tools
from pydap.client import open_url,get_cmr_urls
from pydap.client import to_netcdf as dap_to_netcdf

## Search for OPeNDAP URLs

Will need the concept collection ID (ccid), or the DOI of the data of interest at a minimum.

Can pass additional filters to narrow down the search, such as a time range, bounding box, etc.


In [ ]:
swot_ccid = "C2799438313-POCLOUD"
time_range = [dt.datetime(2023, 2, 1), dt.datetime(2023, 2, 12)] # One month of data

# select a region in the Iceland-Faroe Ridge
bbox = [-20.760731,60.080727, -4.297294,65.675099] #[west, south, east, north]

cmr_urls = get_cmr_urls(ccid=swot_ccid, bounding_box = bbox, time_range=time_range, limit=1000) # you can incread the limit of results
print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

## EDL Authentication

This is a necessary step to download any data, which includes metadata, from a remote file.

There are many ways to authenticate with OPeNDAP in Python, below we use earthaccess as it can abstract the authentication to use tokens to retrieve data.

In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session=auth.get_session()

### Read entire metadata

Reading metadata from one granule within a Collection is important because it allows one to **identify variables of interest to subset by variable name**. OPeNDAP supports it via the use of Constraint Expressions. PyDAP can construct DAP4 constraint expressions when the user provides a list of variable names. 

### How to provide list of names for subset using the DAP4 protocol?

Many of NASA files are hierarchical, meaning that there can be one of more Groups inside the file, and DAP4 fully supports it. 

What is a Group? A Group acts like a folder inside a computer filesystem, and to correctly identifying a variable by its name, one neeed to specify both path and name of the variable. This helps avoid **name collisions**. A name that integrates the full path to the variable is the `Fully Qualifying Name`.



In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

In [ ]:
Variables = [
    "/data_01/time","/data_01/longitude", "/data_01/latitude", 
     "/data_01/surface_classification_flag", "/data_01/ice_flag",
     "/data_01/mean_dynamic_topography", "/data_01/rain_flag"
]
output_path = "data/"

In [ ]:
Variables

# Download all data

Stream each remote file into its own local file, downloading only the data of interest

In [ ]:
%%time
dap_to_netcdf(cmr_urls[:20], session=my_session, keep_variables=Variables, output_path=output_path)

# Inspect a local file

In [ ]:
dt1 = xr.open_datatree(output_path+f"{cmr_urls[0].split('/')[-1]}.nc4")

In [ ]:
dt1